# Lab 09-1 - Ensemble Methods and Random Forests

This lab covers the general approach of ensemble learning (voting ensembles), as well as the specific case of Random Forests models using `sklearn`.

In [ ]:
# Standard Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

# turn off KNN future warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)  

Examples throughout this lab will use the Wisconsin breast cancer data that were used in several of our previous labs:

In [ ]:
# Read the data
wbc = pd.read_csv(filepath_or_buffer="../data/wisc_bc.csv")

# Train-test split
from sklearn.model_selection import train_test_split
train, test = train_test_split(wbc, test_size=0.2, random_state=7)

# Separate the target from the predictors and re-label the target:
# y = 1: 'M' for malignant i.e. cancerous
# y = 0: 'B' for benign i.e. non-cancerous

train_y = train['Label'].map(arg={'M': 1, 'B': 0})
train_X = train.drop(labels=['ID','Label'], axis=1)

test_y = test['Label'].map(arg={'M': 1, 'B': 0})
test_X = test.drop(labels=['ID','Label'], axis=1)

## A) Voting Ensembles

The fundamental idea of an ensemble learner is to combine the prediction rules from several different models/estimators to produce a composite model with superior performance on new data. This can work well if the different models used in the ensemble are each able to learn different things from the training data. Ensembles are most effective when each base model is prone to making different types of errors on the training data.

The simplest kind of ensemble is a *voting ensemble*: train several different models *independently* on the same training data, then combine their predictions by voting (classification) or averaging (regression). Two other kinds of ensembles build on this idea:

- Random forests (later in this lab) - Train many copies of the *same* type of model (a decision tree) on bootstrapped samples of the data.
- Boosting (discussed next time) - Train several models *sequentially* such that each model addresses a weak point of the previous model.

In this section, we'll build a voting ensemble of three types of models using the `VotingClassifier()` function since $y$ is categorical:

1. Logistic regression. Note: `penalty=None` gives plain logistic regression, as in SDS 291. We'll learn more about `penalty` later during regularization
1. Decision Trees. Note: Standardization isn't necessary here because the splits themselves won't change, only their rescaled values will
1. KNN

In [ ]:
# Import pipeline tools
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Import models we've seen in this course
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# For aggregating the results of several models
from sklearn.ensemble import VotingClassifier

# Defining the individual models:
# 1. Logistic regression
# 2. Decision Trees
# 3. KNN
model1 = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(penalty=None))
])
model2 = DecisionTreeClassifier(max_depth=5, random_state=7)
model3 = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier())
])

# Create the ensemble aggregator which aggregates the results of all three:
my_ensemble = VotingClassifier(
    estimators=[('logit', model1), ('tree', model2), ('knn', model3)],
    voting='soft'
)

This example will combine the predictions of the three different classification models using "soft" voting: 

- `voting = 'soft'` will sum the predicted probabilities for each outcome class produced by each model in the ensemble with the final prediction being the class with the largest sum.
- `voting = 'hard'` will use a simple majority vote where the final prediction is the most commonly predicted class. Unfortunately, if there's a tie [the final prediction will be assigned in ascending order](https://scikit-learn.org/stable/modules/ensemble.html#majority-class-labels-majority-hard-voting).

Thus, we'll generally prefer `soft` voting unless our scenario involves a sufficient number of models and outcome classes that make ties unlikely to occur.

Once it is set up, our voting classifier `my_ensemble` has the same capabilities as individual modeling functions in `sklearn`. Ex: we can find the ensemble's cross-validated F1-score and compare it to those of its individual base models. F1-score is an appropriate metric for classifier performance when there is a strong class imbalance and F1-scores closer to 1 indicate good performance, while closer to 0 indicate worse performance. Note however in our case 37.3% of `Label` are `'M'` malignant, which is only mild imbalance. 

In [ ]:
# Import the function for computing cross-validated scores
from sklearn.model_selection import cross_val_score

# Cross-validated F1 score of each model
logistic_scores = cross_val_score(estimator=model1, X=train_X, y=train_y, cv=5, scoring='f1')
print("Logistic regression: " + str(logistic_scores.mean()))

tree_scores = cross_val_score(estimator=model2, X=train_X, y=train_y, cv=5, scoring='f1')
print("Decision tree: " + str(tree_scores.mean()))

knn_scores = cross_val_score(estimator=model3, X=train_X, y=train_y, cv=5, scoring='f1')
print("KNN: " + str(knn_scores.mean()))

ensemble_scores = cross_val_score(estimator=my_ensemble, X=train_X, y=train_y, cv=5, scoring='f1')
print("Ensemble: " + str(ensemble_scores.mean()))

This example illustrates the main motivation for using an ensemble: combining models can give out-of-sample performance that is competitive with, or better than, the individual models. The scores printed above are the cross-validation estimates of each model's F1-score on new unseen data. Here the ensemble is close to its strongest member (logistic regression) and clearly better than the decision tree, so it spares us from having to know in advance which single model will work best. However, it will not always beat every one of its members.

Next, ensembles created using `VotingClassifier()` are fully compatible with pipelines and grid search to optimize the hyperparameters of *individual models*:

In [ ]:
# Import the function for doing a grid search over crossvalidated models
from sklearn.model_selection import GridSearchCV

# 3 x 3 x 2 x 2 tuning parameters to search over. Notice
# 1. Which model in my_ensemble we are tuning over. Ex: knn
# 2. Which tuning parameter we are considering (double underscore precedes it). Ex: __model__n_neighbors
params = {
    'tree__max_depth': [4,5,6],
    'knn__model__n_neighbors': [5,10,15],
    'knn__model__weights': ['distance','uniform'],
    'voting': ['soft','hard']
}

# Perform/fit the grid search
grid = GridSearchCV(
    estimator=my_ensemble,
    param_grid=params,
    cv=5,
    scoring='f1'
).fit(X=train_X, y=train_y)

# Print the best parameter combination and its associated F1-score
# (larger values are better)
print(grid.best_params_)
print("\n")
print("The best F1 score: " + str(grid.best_score_))

The best ensemble model combines

- A logistic regression model
- A KNN model with k=10 using: distance weighting
- A decision tree with max depth 6

aggregated using hard voting

## B) Question 1

- **Part a)**: Create a "soft" voting ensemble involving four base models. Include a standardization step as part of the logistic regression and KNN base models. 
    1. A logistic regression
    1. A decision tree with a max depth of 4
    1. A KNN classifier using $k=5$
    1. A KNN classifier using $k=15$.
- **Part b)**: Conduct a cross-validated grid search using the default scoring metric (classification accuracy) to optimize only the following parameter:
    - The weighting scheme in each KNN base model as either inverse-distance weighting or uniform weighting
- **Part c)**: In a markdown block describe the elements of the best ensemble method and their associated tuning parameters

In [ ]:
# Code for Q1.a)



In [ ]:
# Code for Q1.b)



**Solutions for Q1.c)**

## C) Voting Ensembles for Regression

Predicting a numeric outcome using ensembles is largely the same as the classification examples provided in the previous sections with only a few minor differences. First, you should note that the relevant function is `VotingRegressor()`. By default, the final predictions of `VotingRegressor()` are a simple average of the predictions made by each individual base model. 

## D) Question 2

For this question you should use the Iowa City home sales data provided below. You will use the numeric features in the data to predict the outcome 'sale.amount'.

- **Part a)**: Create an ensemble that uses three base models, being sure to include standardization when needed:
    - A linear regression
    - A decision tree regressor
    - A KNN regressor
- **Part b)**: Perform a cross-validated grid search with the `'neg_root_mean_squared_error'` scoring metric to optimize the following tuning parameters within these base models. Save the result of the `GridSearchCV()` in `grid_q2`.
    - Decision tree depth: either 3, 4, or 5
    - KNN number of neighbors: either 10 or 25
- **Part c)**: In a markdown block describe the elements of the best ensemble method and their associated tuning parameters

In [ ]:
# Read the IC home sales data
ic = pd.read_csv(filepath_or_buffer="../data/IowaCityHomeSales.csv")

# Split to training and testing sets
from sklearn.model_selection import train_test_split
train_ic, test_ic = train_test_split(ic, test_size=0.2, random_state=7)

# Split the outcome from the predictors
train_y_ic = train_ic['sale.amount']
train_X_ic = train_ic.select_dtypes(include="number").drop(labels='sale.amount', axis=1)

print(train_y_ic.head())
train_X_ic.head()

In [ ]:
# Code for Q2.a)
from sklearn.ensemble import VotingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor



In [ ]:
# Code for Q2.b)



**Solutions for Q2.c)**

## E) Random Forests

The random forest algorithm is a popular ensemble approach that aggregates the results of many decision trees that are trained independently on [bootstrapped samples](https://en.wikipedia.org/wiki/Bootstrapping_(statistics)) of the training data. Usually, these trees are forced to use only a subset of the predictors available in the original data set. Together, bootstrapping and feature sampling address two major weaknesses of single decision trees:

1. Deep decision trees are often necessary to adequately capture the predictive patterns in the training data; but deep trees exhibit *high variability*, meaning the splits are highly dependent on the specific set of training data and are prone to overfitting.
2. The decision tree algorithm is *greedy*, so variables that are more predictive will dominate correlated variables that are less predictive but still contain some useful information about the outcome.

Bootstrapping addresses the first weakness by removing the need for deep trees and rewarding splits that generalize well to variations of the training data. Feature sampling addresses the second weakness by allowing more variables to contribute to the ensemble than would normally contribute to a single decision tree.

Overall, the success of a random forest is tied to its ability to build a diverse set of decision trees using slightly different training data and sets of predictors. Achieving success involves selecting appropriate tuning parameters, the most important being:

- `max_depth` - the maximum depth of the individual trees in the ensemble
- `min_samples_split` - the minimum samples in a node for it to be eligible for splitting (for a single tree in the ensemble)
- `max_features` - typically given as an `int` representing the number of randomly selected predictors to be considered in an individual tree in the ensemble.

You should also be aware of the `n_estimators` tuning parameter, which governs the number of trees in the forest. If this parameter is set too low, the forest may not be sufficiently flexible to capture all of the meaningful patterns in the data. However, there is no benefit to choosing a very large number of trees as the performance of a random forest tends to stabilize with a certain number of trees and any more will only add computational burden.

Below is a quick demonstration of `RandomForestClassifier()`. You should note that there is an analogous function `RandomForestRegressor()` that is used for regression tasks.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
my_forest = RandomForestClassifier(
    max_depth=3,
    min_samples_split=10,
    max_features=2,
    n_estimators=200,
    random_state=7
)

We evaluate a random forest the same way as any other model: using cross-validation.

In [ ]:
# Cross-validated classification accuracy
scores = cross_val_score(estimator=my_forest, X=train_X, y=train_y, cv=5)
print(scores.mean())

## F) Question 3

- **Part a)**: Set up a random forest model for the Iowa City home sales data (using all other features to predict `sale.amount`). Set the number of trees to 200 and the random seed. (No standardization needed: trees are unaffected by the scale of the predictors.)
- **Part b)**: Use a cross-validated grid search using the `'neg_root_mean_squared_error'` scoring metric to search over the following tuning parameters. Save the result of the `GridSearchCV()` in `grid_q3`:
    - `max_depth` of either 2 or 3
    - `min_samples_split` of either 20 or 60
    - `max_features` of either 3, 5, or 7
- **Part c)**: Report the best model and its cross-validated score. 
- **Part d)**: Use `cross_val_score()` to compare the best performing random forest from Part b) with the best performing approach from Question 2. Briefly comment upon which method performed better (in terms of its cross-validated RMSE). Hint: you can reuse the best model from Q2 as follows (similarly for Q3):

```
cross_val_score(estimator=grid_q2.best_estimator_, ...)
```


In [ ]:
# Code for Q3.a)
from sklearn.ensemble import RandomForestRegressor



In [ ]:
# Code for Q3.b)



In [ ]:
# Code for Q3.c)



In [ ]:
# Code for Q3.d)

